# Z2005 — Dynamic Programming

A self-study notebook on solving problems with overlapping subproblems and optimal substructure, building from naive recursion to fully tabulated, space-optimized solutions.


## Learning Objectives

By the end of this notebook you should be able to:

- Recognize when a problem has **overlapping subproblems** and **optimal substructure**, the two properties that make dynamic programming (DP) applicable.
- Convert a naive exponential-time recursive solution into a **memoized (top-down)** solution using a cache.
- Convert a memoized solution into a **tabulated (bottom-up)** solution using an explicit table, and explain the tradeoffs between the two.
- Apply DP to solve coin change, 0/1 knapsack, and longest common subsequence (LCS), including reconstructing the actual solution (not just its value).
- Apply **space optimization** (rolling arrays) to reduce a DP solution's memory footprint from O(n) or O(n·W) down to O(1) or O(W) where applicable.
- Explain, with a live `timeit` comparison, why naive recursion is impractical for even moderately sized inputs while memoization and tabulation are not.


## How to Use This Notebook

Run the cells top to bottom. Markdown cells explain the ideas before you see any code — read them, don't skip to the code.

Cells marked `# TODO` are for you to complete; running them before you fill them in will raise `NotImplementedError` on purpose. Cells with `assert` statements are self-checks: they raise an `AssertionError` (with a message telling you what went wrong) if your code is wrong, and print a friendly ✅ message and do nothing else if it is correct. If you get stuck, the fully worked **Solutions** section at the end has the answers — but try each exercise yourself first.


## 1. What Makes a Problem DP-Suitable

Dynamic programming is not a special algorithm so much as a general strategy: instead of solving a big problem directly, you break it into smaller subproblems, solve each one once, and reuse the results. It only pays off when a problem has two specific properties.

**Overlapping subproblems** means that a naive recursive solution ends up solving the *exact same* subproblem many times. Computing `fib(30)` naively recomputes `fib(28)` twice, `fib(27)` three times, and so on — the same calls repeat at an exponential rate. If every subproblem in your recursion tree is genuinely distinct (no repeats), DP has nothing to save you: divide-and-conquer algorithms like merge sort have no overlapping subproblems, and adding a cache to them would only waste memory.

**Optimal substructure** means that an optimal solution to the whole problem can be built from optimal solutions to its subproblems. For shortest paths, the shortest path from A to C through B is the shortest A-to-B path plus the shortest B-to-C path — you never need a "second-best" sub-path to build the best overall path. Not every problem has this: the *longest simple path* in a graph does **not** have optimal substructure, because the longest sub-path might revisit a vertex you already used, which is disallowed in the full path.

A common pitfall is applying DP to a problem that only has one of the two properties. Overlapping subproblems without optimal substructure means memoizing won't get you the right answer, just a faster wrong one. Optimal substructure without overlapping subproblems means memoizing is correct but pointless — you were never going to repeat any work anyway.


In [ ]:
# A quick demonstration: how many times does naive fib() call itself on the same input?
call_counts = {}

def fib_counting(n):
    call_counts[n] = call_counts.get(n, 0) + 1   # record every call, including repeats
    if n <= 1:
        return n
    return fib_counting(n - 1) + fib_counting(n - 2)

fib_counting(10)
# fib(2) alone should be called many times inside the recursion tree for fib(10)
print("fib(2) was computed", call_counts[2], "separate times while computing fib(10)")
print("Total distinct subproblem values seen:", len(call_counts))
print("Total calls made:", sum(call_counts.values()), "— far more than the", len(call_counts), "distinct subproblems")


## 2. Memoization (Top-Down) vs. Tabulation (Bottom-Up)

Both techniques store subproblem results so they're computed once, but they differ in direction. **Memoization** keeps the natural recursive structure of the naive solution and adds a cache (usually a dictionary): before doing the work for `n`, check if the answer is already cached; if not, compute it recursively and store it before returning. It reads almost exactly like the naive version, just with a cache check bolted on.

**Tabulation** flips the direction: instead of starting from the big problem and recursing down, you start from the smallest subproblems and build a table up to the answer you want, using a loop instead of recursion. It avoids function-call overhead entirely and — because Python has a recursion limit (typically 1000) — sidesteps `RecursionError` for large inputs that memoized recursion would hit.

Neither is strictly "better." Memoization is usually easier to derive directly from the recursive definition of a problem and only computes the subproblems actually needed (useful when most of the subproblem space is never visited). Tabulation is usually faster in practice (no call-stack overhead) and easier to space-optimize, but it computes every subproblem up to the target even if some are never needed for the final answer.

A common pitfall with memoization: forgetting to check the cache *before* recursing (so you still branch into both recursive calls first) — you'd merely delay the check rather than avoid the repeated work. Another common pitfall with tabulation: getting the fill order wrong so that a table cell is read before it's written.


In [ ]:
# Fibonacci: naive, memoized (top-down), and tabulated (bottom-up)

def fib_naive(n):
    if n <= 1:
        return n
    # Recomputes fib_naive(n-2) inside both branches below — this is the
    # overlapping-subproblem waste we just measured above.
    return fib_naive(n - 1) + fib_naive(n - 2)

def fib_memo(n, cache=None):
    if cache is None:
        cache = {}
    if n in cache:                       # cache HIT: skip the recursion entirely
        return cache[n]
    if n <= 1:
        result = n
    else:
        result = fib_memo(n - 1, cache) + fib_memo(n - 2, cache)
    cache[n] = result                    # store before returning so future calls hit the cache
    return result

def fib_tabulated(n):
    if n <= 1:
        return n
    table = [0] * (n + 1)                # table[i] will hold fib(i)
    table[1] = 1
    for i in range(2, n + 1):            # bottom-up: fill smallest subproblems first
        table[i] = table[i - 1] + table[i - 2]
    return table[n]

# All three must agree on small inputs
assert fib_naive(10) == fib_memo(10) == fib_tabulated(10) == 55
# Naive would take far too long here; memoized and tabulated must still agree
assert fib_memo(50) == fib_tabulated(50) == 12586269025
print("Fibonacci checks passed: naive, memoized, and tabulated all agree")


### Live timing comparison

Talking about "naive recursion is too slow" is one thing; seeing it is more convincing. The cell below times `fib_naive` against `fib_memo` and `fib_tabulated` for a moderately large `n` using `timeit`, which runs each version several times and reports real elapsed time — no invented numbers, whatever your machine produces is what gets printed.


In [ ]:
import timeit

n_small = 27   # small enough that naive recursion finishes in reasonable time, large enough to show a gap

naive_time = timeit.timeit(lambda: fib_naive(n_small), number=3)
memo_time = timeit.timeit(lambda: fib_memo(n_small), number=3)
tab_time = timeit.timeit(lambda: fib_tabulated(n_small), number=3)

print(f"fib_naive({n_small}):     {naive_time:.4f}s for 3 runs")
print(f"fib_memo({n_small}):      {memo_time:.4f}s for 3 runs")
print(f"fib_tabulated({n_small}): {tab_time:.4f}s for 3 runs")

# The naive version does exponentially more work, so it should be measurably
# slower than both DP versions on this input (the DP versions should each
# finish in comfortably under a tenth of a second).
assert naive_time > memo_time
assert naive_time > tab_time
print("\nConfirmed live: naive recursion is measurably slower than either DP version")


## 3. Space Optimization: The Rolling Array

Look again at `fib_tabulated`: to compute `table[i]` you only ever need `table[i-1]` and `table[i-2]`. Keeping the *entire* table of size `n+1` around is wasteful — you can keep just the last two values and "roll" them forward each iteration. This turns O(n) space into O(1) space, at the cost of no longer being able to look back at earlier values (which is fine if you only need the final answer, not the full history).

The same idea generalizes to two-dimensional DP tables such as 0/1 knapsack: if row `i` of the table only depends on row `i-1`, you don't need to keep every row — two rows (or, with careful iteration order, a single row updated in place) are enough. The tradeoff is that reconstructing *which* choices were made (not just the optimal value) usually needs the full table, so space optimization and solution-reconstruction are often in tension. We'll see this tradeoff directly in the knapsack section below.


In [ ]:
def fib_rolling(n):
    if n <= 1:
        return n
    prev2, prev1 = 0, 1                  # prev2 = fib(0), prev1 = fib(1)
    for _ in range(2, n + 1):
        prev2, prev1 = prev1, prev2 + prev1   # roll the window forward by one step
    return prev1

assert fib_rolling(10) == 55
assert fib_rolling(50) == fib_tabulated(50)
print("Rolling-array Fibonacci matches the full-table version, using O(1) space instead of O(n)")


## 4. Coin Change (Fewest Coins to Make a Target)

Given a set of coin denominations and a target amount, find the *minimum number of coins* needed to make that amount exactly (assuming unlimited supply of each denomination). This is a classic 1-D DP problem: `f[a]` is the fewest coins needed to make amount `a`, and it's built from smaller amounts: `f[a] = 1 + min(f[a - c])` over every coin `c` that's no larger than `a`.

A frequent instinct is to reach for a **greedy** approach instead: always take the largest coin that fits. Greedy is correct for "nice" currency systems (like US coins: 1, 5, 10, 25) but is *not* correct in general. With coins `[1, 3, 4]` and a target of 6, greedy picks `4 + 1 + 1` (three coins), but the true optimum is `3 + 3` (two coins). DP explores every combination systematically and is guaranteed correct regardless of which denominations you're given; greedy is only a shortcut that happens to work for specific coin systems.

The pitfall to watch for is initializing `f[a]` incorrectly: amounts that cannot be formed at all need to be represented (e.g. with infinity or `None`) rather than left as 0, otherwise the algorithm will think an unreachable amount is free.


In [ ]:
def coin_change_min_count(coins, target):
    # f[a] = fewest coins to make amount a; float('inf') means "not yet known to be reachable"
    f = [0] + [float('inf')] * target
    for a in range(1, target + 1):
        for c in coins:
            if c <= a and f[a - c] + 1 < f[a]:
                f[a] = f[a - c] + 1
    return f[target]

def coin_change_reconstruct(coins, target):
    # Same recurrence as above, but also remembers which coin achieved the best count
    f = [0] * (target + 1)
    choice = [None] * (target + 1)
    for a in range(1, target + 1):
        best = float('inf')
        for c in coins:
            if c <= a and 1 + f[a - c] < best:
                best = 1 + f[a - c]
                choice[a] = c
        f[a] = best
    result, a = [], target
    while a > 0:                         # walk the choice trail backwards to list the coins used
        result.append(choice[a])
        a -= choice[a]
    return f[target], result

def greedy_coin_change(coins, target):
    coins = sorted(coins, reverse=True)  # always try the biggest coin first
    count = 0
    for c in coins:
        count += target // c
        target %= c
    return count

# DP gives the true optimum for [1, 3, 4], target 6: two 3-coins
value, coins_used = coin_change_reconstruct([1, 3, 4], 6)
assert value == 2 and sorted(coins_used) == [3, 3]
assert coin_change_min_count([1, 3, 4], 6) == 2

# Greedy is provably wrong on this same input: it picks 4 + 1 + 1 (three coins)
assert greedy_coin_change([1, 3, 4], 6) == 3
print("Coin change: DP found the true optimum (2 coins); greedy was wrong (3 coins) on the same input")


## 5. 0/1 Knapsack

You have a knapsack that can carry total weight `W`, and a set of items, each with a `weight` and a `value`. Each item can be taken at most once (that's the "0/1" — you either take it or you don't; no fractional items, no duplicates). The goal is to choose a subset of items maximizing total value without exceeding `W`.

This is a genuinely two-dimensional DP: the state depends on *both* which items you've considered so far and how much capacity remains. `table[i][w]` is the best value achievable using only the first `i` items with capacity `w`. For each item you have exactly two choices — skip it (`table[i-1][w]`) or take it if it fits (`values[i-1] + table[i-1][w - weights[i-1]]`) — and you take whichever is better.

Getting the *value* of the optimal solution is only half the problem; often you also need to know *which items* were chosen. That requires walking back through the table from `table[n][W]`: if the value differs from `table[i-1][W]`, item `i-1` must have been included, so you subtract its weight and continue. This reconstruction step is why we keep the full 2-D table here rather than space-optimizing it — a rolling 1-D array would give you the optimal value but lose the information needed to trace back which items were picked.


In [ ]:
def knapsack_table(weights, values, W):
    n = len(weights)
    # table[i][w] = best value using the first i items with capacity w
    table = [[0] * (W + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for w in range(W + 1):
            if weights[i - 1] > w:
                table[i][w] = table[i - 1][w]                 # item i-1 doesn't fit: must skip it
            else:
                skip = table[i - 1][w]
                take = values[i - 1] + table[i - 1][w - weights[i - 1]]
                table[i][w] = max(skip, take)                 # better of skipping or taking
    return table

def knapsack_reconstruct(table, weights, n, W):
    chosen = []
    for i in range(n, 0, -1):
        if table[i][W] != table[i - 1][W]:   # value changed => item i-1 must have been taken
            chosen.append(i - 1)
            W -= weights[i - 1]
    return chosen

weights = [1, 3, 4]
values = [15, 20, 30]
table = knapsack_table(weights, values, 4)
assert table[3][4] == 35                     # best value with capacity 4
chosen = knapsack_reconstruct(table, weights, 3, 4)
assert sorted(chosen) == [0, 1]              # items 0 and 1 (weights 1+3=4, values 15+20=35)
print("Knapsack checks passed: optimal value 35, achieved by items", sorted(chosen))


### Space-optimizing the knapsack value (without reconstruction)

If you only need the optimal *value* (not which items achieve it), the 2-D table can be collapsed to a single 1-D array of size `W + 1`, updated in place — but only if you iterate capacity `w` from **high to low** within each item's pass. Iterating low to high would let an item be "reused" within the same pass, turning 0/1 knapsack into the unbounded (repeat-allowed) variant by accident, which is a genuinely common bug.


In [ ]:
def knapsack_value_only(weights, values, W):
    dp = [0] * (W + 1)                   # dp[w] = best value achievable with capacity w so far
    for i in range(len(weights)):
        # Must go high-to-low: otherwise dp[w - weights[i]] might already
        # reflect item i being used earlier in this same pass (0/1 requires
        # each item used at most once).
        for w in range(W, weights[i] - 1, -1):
            dp[w] = max(dp[w], values[i] + dp[w - weights[i]])
    return dp[W]

assert knapsack_value_only(weights, values, 4) == 35   # matches the full 2-D table result above
print("Space-optimized knapsack (O(W) space) matches the full 2-D table: optimal value 35")


## 6. Longest Common Subsequence (LCS)

Given two strings, find the length (and, with reconstruction, the actual content) of the longest sequence of characters that appears in both, in the same relative order, but not necessarily contiguously. `"ABC"` and `"AC"` share the subsequence `"AC"` (length 2) — `A` and `C` appear in that order in both strings, even though `B` sits between them in the first one.

The recurrence compares the strings character by character from the end: if the last characters match, they must both be part of the LCS, so the answer is `1 + LCS` of both strings with that character removed. If they don't match, the LCS is the better of dropping the last character from either string. This gives `L[i][j] = L[i-1][j-1] + 1` when `s1[i-1] == s2[j-1]`, otherwise `L[i][j] = max(L[i-1][j], L[i][j-1])`.

A closely related problem is **edit distance** (Levenshtein distance): the minimum number of single-character insertions, deletions, or substitutions to turn one string into another. It uses the same 2-D table shape but a different recurrence, since every mismatched pair now costs 1 (a substitution) rather than being skipped for free.


In [ ]:
def lcs_table(s1, s2):
    n, m = len(s1), len(s2)
    L = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s1[i - 1] == s2[j - 1]:
                L[i][j] = L[i - 1][j - 1] + 1        # matching character: extend the LCS
            else:
                L[i][j] = max(L[i - 1][j], L[i][j - 1])   # no match: best of dropping from either string
    return L

def lcs_reconstruct(table, s1, s2, i, j):
    if i == 0 or j == 0:
        return ""
    if s1[i - 1] == s2[j - 1]:
        # This character is part of the LCS; recurse on the shorter prefixes and append it
        return lcs_reconstruct(table, s1, s2, i - 1, j - 1) + s1[i - 1]
    elif table[i - 1][j] >= table[i][j - 1]:
        return lcs_reconstruct(table, s1, s2, i - 1, j)
    else:
        return lcs_reconstruct(table, s1, s2, i, j - 1)

L = lcs_table("ABC", "AC")
assert L[3][2] == 2
assert lcs_reconstruct(L, "ABC", "AC", 3, 2) == "AC"

def edit_distance(s1, s2):
    n, m = len(s1), len(s2)
    D = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        D[i][0] = i                      # deleting all i characters of s1 to match empty s2
    for j in range(m + 1):
        D[0][j] = j                      # inserting all j characters to build s2 from empty s1
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s1[i - 1] == s2[j - 1]:
                D[i][j] = D[i - 1][j - 1]                 # matching character costs nothing
            else:
                D[i][j] = 1 + min(D[i - 1][j - 1],        # substitute
                                   D[i - 1][j],            # delete from s1
                                   D[i][j - 1])            # insert into s1
    return D[n][m]

assert edit_distance("kitten", "sitting") == 3
print("LCS and edit distance checks passed:", '"AC"', "is the LCS of ABC/AC; kitten->sitting costs 3 edits")


## Exercises

Work through these in order — each builds on ideas from the sections above. Fill in the `# TODO` in each function; do not change the function signature. Run the matching self-check cell afterwards to confirm your solution.


### Exercise 1 — Climbing Stairs (Fibonacci variant)

You are climbing a staircase with `n` steps. Each move you can climb either 1 or 2 steps. Write `climb_stairs(n)` that returns the number of *distinct* ways to reach the top.

Example: `climb_stairs(3)` returns `3` (the ways are `1+1+1`, `1+2`, `2+1`).

This should feel structurally identical to `fib_tabulated` above — think about what the recurrence `ways[i] = ways[i-1] + ways[i-2]` means here and why.


In [ ]:
def climb_stairs(n: int) -> int:
    """Return the number of distinct ways to climb n stairs, moving 1 or 2 steps at a time.

    n=0 -> 1 (one way: do nothing)
    n=1 -> 1
    n=2 -> 2
    n=3 -> 3
    """
    # TODO: implement this using tabulation (a loop building up a table or
    # rolling variables), not naive recursion.
    raise NotImplementedError


### Exercise 2 — Maximum Subarray Sum

Given a list of integers (which may include negative numbers), write `max_subarray_sum(nums)` that returns the largest possible sum of a *contiguous* subarray.

Example: `max_subarray_sum([-2, 1, -3, 4, -1, 2, 1, -5, 4])` returns `6` (the subarray `[4, -1, 2, 1]`).

Hint: define `best_ending_here[i]` as the best sum of a subarray that *ends* at index `i`. Either extend the previous best subarray by including `nums[i]`, or start fresh at `i` — whichever is larger. This is Kadane's algorithm, and it is DP even though it's often taught separately.


In [ ]:
def max_subarray_sum(nums: list) -> int:
    """Return the maximum sum of any contiguous, non-empty subarray of nums.

    Assumes nums is non-empty.
    """
    # TODO: implement this in O(n) time using a rolling "best subarray ending here" value.
    raise NotImplementedError


### Exercise 3 — Coin Change (Fewest Coins), From Scratch

Write `min_coins(coins, target)` that returns the fewest coins needed to make `target` exactly, or `-1` if it is impossible with the given denominations.

Example: `min_coins([1, 3, 4], 6)` returns `2`. `min_coins([2], 3)` returns `-1` (3 is odd, can't be made from 2s).

This is the same recurrence as `coin_change_min_count` above — write it yourself without looking back, then handle the "impossible" case explicitly.


In [ ]:
def min_coins(coins: list, target: int) -> int:
    """Return the fewest coins from `coins` (unlimited supply each) that sum to target,
    or -1 if target cannot be formed exactly.
    """
    # TODO: implement using tabulation. Remember to detect and return -1 for
    # amounts that stay unreachable (infinity) after filling the table.
    raise NotImplementedError


### Exercise 4 — 0/1 Knapsack Value, Space-Optimized

Write `knapsack_max_value(weights, values, W)` that returns the maximum total value achievable without exceeding capacity `W`, using **only O(W) space** (a single 1-D array), following the rolling-array pattern shown in `knapsack_value_only` above.

Example: `knapsack_max_value([1, 3, 4], [15, 20, 30], 4)` returns `35`.

Pay close attention to the iteration direction over capacity — getting it wrong silently turns this into the *unbounded* knapsack variant instead of 0/1.


In [ ]:
def knapsack_max_value(weights: list, values: list, W: int) -> int:
    """Return the maximum value achievable with total weight <= W, each item usable at most once.

    Must use O(W) space (a single 1-D DP array), not a full 2-D table.
    """
    # TODO: implement using a single 1-D array, iterating capacity from high to low for each item.
    raise NotImplementedError


### Exercise 5 (harder) — Longest Increasing Subsequence

Write `longest_increasing_subsequence(nums)` that returns the length of the longest strictly increasing subsequence of `nums` (elements need not be contiguous, but must appear in increasing order and increasing value).

Example: `longest_increasing_subsequence([10, 9, 2, 5, 3, 7, 101, 18])` returns `4` (one such subsequence is `[2, 3, 7, 18]` or `[2, 3, 7, 101]`).

This is a new recurrence, not a direct copy of anything above: define `lis_ending_at[i]` as the length of the longest increasing subsequence that ends exactly at index `i`. For each `i`, look back at every `j < i` where `nums[j] < nums[i]`, and take the best `lis_ending_at[j] + 1`. The straightforward version of this is O(n^2); that is acceptable here.


In [ ]:
def longest_increasing_subsequence(nums: list) -> int:
    """Return the length of the longest strictly increasing subsequence of nums.

    Assumes nums is non-empty.
    """
    # TODO: implement the O(n^2) DP described above.
    raise NotImplementedError


## Self-Check

Run each cell below after implementing the matching exercise. A passing cell prints a ✅ message and does nothing else; a failing cell raises an `AssertionError` explaining what went wrong.


In [ ]:
# Self-check for Exercise 1
assert climb_stairs(0) == 1, "climb_stairs(0) should be 1 (one way: do nothing)"
assert climb_stairs(1) == 1
assert climb_stairs(2) == 2
assert climb_stairs(3) == 3
assert climb_stairs(5) == 8
print("✅ Exercise 1 passed")


In [ ]:
# Self-check for Exercise 2
assert max_subarray_sum([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6
assert max_subarray_sum([1]) == 1
assert max_subarray_sum([5, 4, -1, 7, 8]) == 23
assert max_subarray_sum([-3, -1, -2]) == -1, "with all-negative input, the answer is the least-negative single element"
print("✅ Exercise 2 passed")


In [ ]:
# Self-check for Exercise 3
assert min_coins([1, 3, 4], 6) == 2
assert min_coins([1, 5, 10, 25], 30) == 2   # a quarter and a nickel
assert min_coins([2], 3) == -1
assert min_coins([1], 0) == 0
print("✅ Exercise 3 passed")


In [ ]:
# Self-check for Exercise 4
assert knapsack_max_value([1, 3, 4], [15, 20, 30], 4) == 35
assert knapsack_max_value([2, 3, 4, 5], [3, 4, 5, 6], 5) == 7
assert knapsack_max_value([10], [100], 5) == 0   # item doesn't fit at all
print("✅ Exercise 4 passed")


In [ ]:
# Self-check for Exercise 5
assert longest_increasing_subsequence([10, 9, 2, 5, 3, 7, 101, 18]) == 4
assert longest_increasing_subsequence([0, 1, 0, 3, 2, 3]) == 4
assert longest_increasing_subsequence([7, 7, 7, 7]) == 1   # strictly increasing: repeats don't extend it
print("✅ Exercise 5 passed")


## Quiz

Try to answer each question yourself before revealing the answer.

**1. Why does naive recursive Fibonacci run in exponential time, while memoized Fibonacci runs in linear time, even though they compute the exact same recurrence?**

<details><summary>Show answer</summary>
The naive version's recursion tree has O(2^n) nodes because it recomputes the same subproblem values many times (e.g. fib(n-2) is reached via two different paths). Memoization caches each distinct subproblem's result the first time it's computed, so every subsequent call to the same subproblem is O(1); since there are only n+1 distinct subproblems (fib(0) through fib(n)), the total work is O(n).
</details>

**2. In the 0/1 knapsack space-optimized version, why must the inner loop over capacity `w` go from high to low rather than low to high?**

<details><summary>Show answer</summary>
Because the array is updated in place. If you iterate low to high, then when updating dp[w] for a larger w you might read dp[w - weights[i]], which could already include item i counted earlier in the same pass over w — effectively letting item i be used twice. Iterating high to low guarantees that dp[w - weights[i]] still reflects only items processed before item i, preserving the 0/1 (use-at-most-once) constraint.
</details>

**3. Greedy coin change works for US currency (1, 5, 10, 25) but fails for coins [1, 3, 4] with target 6. What property of a coin system determines whether greedy is safe to use?**

<details><summary>Show answer</summary>
Roughly, a coin system is "greedy-safe" (also called canonical) if every value can be optimally represented by always taking the largest coin that fits, which holds when each denomination is a large enough multiple of the ones below it, with no gaps that force worse combinations. There's no simple universal rule that covers every coin system, though; the only way to be certain a specific system is greedy-safe is to prove it or exhaustively check it (as the coin-change section above does for US currency across every target from 0 to 100), or simply always use DP, which is correct regardless.
</details>

**4. Why does 0/1 knapsack need a full 2-D table for solution reconstruction, but only a 1-D array if you just need the optimal value?**

<details><summary>Show answer</summary>
Reconstructing which items were chosen requires comparing table[i][w] against table[i-1][w] at every step to determine whether item i-1 was included, which means you need every row (every "items considered so far" state) still available after the fact. The space-optimized 1-D version overwrites earlier states as it goes, so by the time you reach the final answer, the information about which specific items produced it has already been discarded — only the numeric value survives.
</details>


## Solutions (try the exercises yourself first!)

Everything below is fully solved. If you're still working through the exercises, stop scrolling.


In [ ]:
# Solution 1 — Climbing Stairs
def climb_stairs_solution(n: int) -> int:
    if n <= 1:
        return 1
    prev2, prev1 = 1, 1          # ways(0) = 1, ways(1) = 1
    for _ in range(2, n + 1):
        prev2, prev1 = prev1, prev2 + prev1
    return prev1

assert climb_stairs_solution(0) == 1
assert climb_stairs_solution(5) == 8
print("Solution 1 verified")


In [ ]:
# Solution 2 — Maximum Subarray Sum (Kadane's algorithm)
def max_subarray_sum_solution(nums: list) -> int:
    best_ending_here = nums[0]
    best_overall = nums[0]
    for x in nums[1:]:
        best_ending_here = max(x, best_ending_here + x)   # extend or restart
        best_overall = max(best_overall, best_ending_here)
    return best_overall

assert max_subarray_sum_solution([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6
assert max_subarray_sum_solution([-3, -1, -2]) == -1
print("Solution 2 verified")


In [ ]:
# Solution 3 — Coin Change (Fewest Coins)
def min_coins_solution(coins: list, target: int) -> int:
    f = [0] + [float('inf')] * target
    for a in range(1, target + 1):
        for c in coins:
            if c <= a and f[a - c] + 1 < f[a]:
                f[a] = f[a - c] + 1
    return f[target] if f[target] != float('inf') else -1

assert min_coins_solution([1, 3, 4], 6) == 2
assert min_coins_solution([2], 3) == -1
print("Solution 3 verified")


In [ ]:
# Solution 4 — 0/1 Knapsack Value, Space-Optimized
def knapsack_max_value_solution(weights: list, values: list, W: int) -> int:
    dp = [0] * (W + 1)
    for i in range(len(weights)):
        for w in range(W, weights[i] - 1, -1):
            dp[w] = max(dp[w], values[i] + dp[w - weights[i]])
    return dp[W]

assert knapsack_max_value_solution([1, 3, 4], [15, 20, 30], 4) == 35
print("Solution 4 verified")


In [ ]:
# Solution 5 — Longest Increasing Subsequence
def longest_increasing_subsequence_solution(nums: list) -> int:
    n = len(nums)
    lis_ending_at = [1] * n          # every single element is an LIS of length 1 by itself
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                lis_ending_at[i] = max(lis_ending_at[i], lis_ending_at[j] + 1)
    return max(lis_ending_at)

assert longest_increasing_subsequence_solution([10, 9, 2, 5, 3, 7, 101, 18]) == 4
assert longest_increasing_subsequence_solution([7, 7, 7, 7]) == 1
print("Solution 5 verified")


## MTech Extension — Bitmask DP for the Traveling Salesman Problem (Held-Karp)

Everything above generalizes: DP state doesn't have to be a single index or a pair of indices. A powerful extension is **bitmask DP**, where the state includes a bitmask representing a *subset* of items already processed — useful whenever the order or combination of a small set of elements matters.

The classic example is the Traveling Salesman Problem (TSP): given `n` cities and pairwise distances, find the shortest route that visits every city exactly once and returns to the start. Brute force checks all `(n-1)!` orderings. The **Held-Karp algorithm** uses DP over `(subset, last_city)` pairs: `dp[mask][j]` is the shortest path that visits exactly the cities in `mask`, ending at city `j`. This is O(n^2 * 2^n) — still exponential, but a dramatic improvement over O(n!) (for n=15, that's roughly 15^2 * 2^15 ≈ 7.4 million versus 15! ≈ 1.3 trillion).

Discussion point for MTech: bitmask DP is only practical while `n` is small (roughly n <= 20, since 2^20 is about a million and the table has n * 2^n entries). For larger instances, exact DP is abandoned in favor of approximation algorithms or metaheuristics (e.g. nearest-neighbor construction with 2-opt local search, or genetic algorithms) — worth having students articulate *why* the exponential state space, not just runtime, is the hard limit here, and that this is a fundamentally different kind of intractability than "just needs a faster constant factor."


In [ ]:
from itertools import combinations

def tsp_held_karp(dist):
    """Solve TSP exactly via Held-Karp bitmask DP.

    dist: n x n matrix, dist[i][j] = distance from city i to city j.
    Returns the length of the shortest tour starting and ending at city 0,
    visiting every other city exactly once.
    """
    n = len(dist)
    if n == 1:
        return 0

    # dp[mask][j] = shortest path visiting exactly the cities in `mask`,
    # starting at city 0, ending at city j (j must be in mask).
    # We only need masks that include city 0 and j; represent city 0 as bit 0.
    FULL = 1 << n
    dp = [[float('inf')] * n for _ in range(FULL)]
    dp[1][0] = 0   # mask with only city 0 visited, ending at city 0, costs 0

    for mask in range(FULL):
        if not (mask & 1):          # every valid mask must include the start city (bit 0)
            continue
        for j in range(n):
            if not (mask & (1 << j)) or dp[mask][j] == float('inf'):
                continue            # j not in this subset, or this state is unreachable
            for k in range(n):
                if mask & (1 << k):
                    continue        # k already visited in this subset
                next_mask = mask | (1 << k)
                candidate = dp[mask][j] + dist[j][k]
                if candidate < dp[next_mask][k]:
                    dp[next_mask][k] = candidate

    full_mask = FULL - 1
    # Close the tour: return from the last city back to city 0
    return min(dp[full_mask][j] + dist[j][0] for j in range(1, n))

def tsp_brute_force(dist):
    """Exhaustive check, only for verifying Held-Karp on small instances."""
    n = len(dist)
    if n == 1:
        return 0
    best = float('inf')
    for perm in __import__('itertools').permutations(range(1, n)):
        route = [0] + list(perm)
        total = sum(dist[route[i]][route[i + 1]] for i in range(n - 1)) + dist[route[-1]][0]
        best = min(best, total)
    return best

# A small 5-city instance; verify Held-Karp against brute force
dist_matrix = [
    [0, 2, 9, 10, 7],
    [1, 0, 6, 4, 3],
    [15, 7, 0, 8, 3],
    [6, 3, 12, 0, 11],
    [9, 5, 4, 6, 0],
]

held_karp_result = tsp_held_karp(dist_matrix)
brute_force_result = tsp_brute_force(dist_matrix)
assert held_karp_result == brute_force_result
print(f"Held-Karp shortest tour: {held_karp_result} (matches brute force: {brute_force_result})")

# Sanity check on state-space growth: table size is n * 2^n
for n_cities in [5, 10, 15, 20]:
    table_size = n_cities * (2 ** n_cities)
    print(f"n={n_cities:2d} cities -> dp table has {table_size:,} entries")
